In [2]:
import pandas as pd
from difflib import SequenceMatcher
from datetime import timedelta

# 1. Load file kết quả vừa chạy xong (để xử lý tiếp các ca Miss)
# Lưu ý: Sửa tên file nếu bro lưu khác
df_current = pd.read_csv('result4/merged_xray_smart_match.csv') 
df_ketqua = pd.read_csv('ketqua.csv')

print(f"Tổng số record: {len(df_current)}")
print(f"Số record đã match: {len(df_current[df_current['match_status'] == 'Matched'])}")
print(f"Số record cần giải cứu (Miss): {len(df_current[df_current['match_status'] != 'Matched'])}")

# --- CHUẨN BỊ DỮ LIỆU ---
# Cần convert lại datetime vì khi save/load CSV nó về dạng string
df_current['datetime_obj'] = pd.to_datetime(df_current['datetime'], errors='coerce')
df_current['date_only'] = df_current['datetime_obj'].dt.date

df_ketqua['date_start_obj'] = pd.to_datetime(df_ketqua['Date start'], format='%d/%m/%Y %H:%M', errors='coerce')
df_ketqua['date_only'] = df_ketqua['date_start_obj'].dt.date

# Hàm chuẩn hóa (dùng lại logic cũ)
import unicodedata
def normalize_text(text):
    if pd.isna(text): return ""
    text = str(text).upper()
    text = unicodedata.normalize('NFKD', text).encode('ASCII', 'ignore').decode('utf-8')
    return text.strip()

df_ketqua['norm_name'] = df_ketqua['Name'].apply(normalize_text)
# df_current đã có cột norm_name từ lần chạy trước, không cần làm lại

# --- HÀM TÌM KIẾM MỞ RỘNG (RESCUE) ---
def find_rescue_match(row_xray, df_lookup):
    if pd.isna(row_xray['date_only']):
        return None, 0, "No Date"

    xray_date = row_xray['date_only']
    xray_name = str(row_xray['norm_name'])
    
    # CHIẾN LƯỢC 2: Mở rộng +/- 3 ngày
    start_window = xray_date - timedelta(days=3)
    end_window = xray_date + timedelta(days=3)
    
    # Lọc ứng viên trong 7 ngày (3 trước, 3 sau, 1 hiện tại)
    candidates = df_lookup[
        (df_lookup['date_only'] >= start_window) & 
        (df_lookup['date_only'] <= end_window)
    ]
    
    if candidates.empty:
        return None, 0, "No Candidates (+/-3 days)"

    best_score = 0
    best_idx = None

    for idx, cand in candidates.iterrows():
        cand_name = cand['norm_name']
        score = SequenceMatcher(None, xray_name, cand_name).ratio()
        
        if score > best_score:
            best_score = score
            best_idx = idx
    
    # Threshold: Vì đã mở rộng ngày (rủi ro cao hơn), ta cần tên phải giống tương đối
    # Giữ mức 0.65 hoặc 0.7 để tránh match nhầm người khác trong mấy ngày đó
    if best_score > 0.65:
        return best_idx, best_score, "Matched (Rescue +/-3d)"
    else:
        return None, best_score, "Low Score Rescue"

# --- CHẠY VÒNG GIẢI CỨU ---
print("\nĐang chạy vòng giải cứu (Rescue Phase)...")

# Chỉ iter qua những dòng chưa match
mask_miss = df_current['match_status'] != 'Matched'
indices_to_rescue = df_current[mask_miss].index

count = 0
rescued_count = 0

for i in indices_to_rescue:
    row = df_current.loc[i]
    
    match_idx, score, status = find_rescue_match(row, df_ketqua)
    
    if match_idx is not None:
        # Cập nhật trực tiếp vào DataFrame gốc
        df_current.at[i, 'match_score'] = score
        df_current.at[i, 'match_status'] = status
        
        # Lấy data từ ketqua đắp vào
        match_data = df_ketqua.loc[match_idx]
        for col in df_ketqua.columns:
            # Chỉ lấy các cột data gốc của ketqua (tránh mấy cột temp)
            if col not in ['norm_name', 'date_start_obj', 'date_only', 'calc_birth_year']:
                target_col = f"ketqua_{col}"
                # Nếu cột chưa tồn tại trong df_current thì tạo mới (ít xảy ra vì đã có từ pass 1)
                df_current.at[i, target_col] = match_data[col]
        
        rescued_count += 1
    
    count += 1
    if count % 500 == 0:
        print(f"Đã duyệt {count} ca miss...")

# --- KẾT QUẢ CUỐI CÙNG ---
final_matched = len(df_current[df_current['match_status'].str.contains('Matched')])
final_rate = (final_matched / len(df_current)) * 100

print("-" * 30)
print(f"SỐ CA ĐƯỢC CỨU THÊM: {rescued_count}")
print(f"Tổng số record: {len(df_current)}")
print(f"Tổng số Match cuối cùng: {final_matched}")
print(f"Tỷ lệ Match cuối cùng: {final_rate:.2f}%")
print("-" * 30)

# Lưu file version 2
df_current.to_csv('result4/merged_xray_final_v2.csv', index=False)
print("Đã lưu file: merged_xray_final_v2.csv")

Tổng số record: 38486
Số record đã match: 35645
Số record cần giải cứu (Miss): 2841

Đang chạy vòng giải cứu (Rescue Phase)...
Đã duyệt 500 ca miss...
Đã duyệt 1000 ca miss...
Đã duyệt 1500 ca miss...
Đã duyệt 2000 ca miss...
Đã duyệt 2500 ca miss...
------------------------------
SỐ CA ĐƯỢC CỨU THÊM: 570
Tổng số record: 38486
Tổng số Match cuối cùng: 36215
Tỷ lệ Match cuối cùng: 94.10%
------------------------------
Đã lưu file: merged_xray_final_v2.csv


In [4]:
import pandas as pd

# 1. Load file kết quả cuối cùng (v2)
df_final = pd.read_csv('result4/merged_xray_final_v2.csv')

# 2. Lọc ra những dòng chưa match (Match status khác 'Matched...')
# Lưu ý: Status có thể là 'Matched' hoặc 'Matched (Rescue +/-3d)' nên ta lọc những cái KHÔNG chứa chữ 'Matched'
missed_df = df_final[~df_final['match_status'].str.contains('Matched', na=False)]

# 3. Chọn các cột quan trọng để dễ soi
cols_to_view = ['filepath', 'patient_name', 'birth_year', 'datetime', 'match_score', 'match_status']
# Nếu file có nhiều cột, ta chỉ lấy những cột này cho gọn
missed_view = missed_df[cols_to_view]

# 4. Xuất ra file
output_miss = 'result4/unmatched_records_review.csv'
missed_view.to_csv(output_miss, index=False)

print(f"Đã xuất {len(missed_view)} bản ghi chưa match ra file: {output_miss}")
print("Bạn hãy mở file này lên để kiểm tra. Nếu thấy lỗi OCR (ví dụ tên là '???'), thì đó là lý do không thể match.")

# 5. Xóa các ca chưa match khỏi file kết quả cuối cùng để chỉ giữ lại các ca đã match
df_matched_only = df_final[df_final['match_status'].str.contains('Matched', na=False)]
df_matched_only.to_csv('result4/dataset.csv', index=False)
print("Đã lưu file chỉ chứa các ca đã match")

Đã xuất 2271 bản ghi chưa match ra file: result4/unmatched_records_review.csv
Bạn hãy mở file này lên để kiểm tra. Nếu thấy lỗi OCR (ví dụ tên là '???'), thì đó là lý do không thể match.
Đã lưu file chỉ chứa các ca đã match
